https://github.com/ventolab/CellphoneDB/tree/master

NOTE: MUST USE HUMAN GENE NAMES, MAKE NEW OBJECT WITH THE SAME METHOD AS MOUSE GENE NAMES

In [3]:
import anndata  
import pandas as pd
import anndata as ad
import seaborn as sb
import scanpy as sc
import cellphonedb
import glob
import os
import sys

ImportError: cannot import name '_errors' from partially initialized module 'h5py' (most likely due to a circular import) (/opt/anaconda3/lib/python3.12/site-packages/h5py/__init__.py)

In [2]:
obj = anndata.io.read_h5ad('C:/Users/Gabe/Desktop/RNA_object_anndata.h5ad')
###IT WORKED

In [3]:
obj.obs.columns

Index(['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nCount_ATAC',
       'nFeature_ATAC', 'TSS.enrichment', 'TSS.percentile',
       'peak_region_fragments', 'log10nfragments', 'pct_reads_in_peaks',
       'nucleosome_signal', 'nucleosome_percentile', 'individual', 'pct.mt',
       'pct.rb', 'pct.mt.rb', 'pct.cd', 'pct.nuclear', 'pct.ambient',
       'log10GenesPerUMI', 'log10PeaksPerUMI', 'genes_per_umi',
       'peaks_per_umi', 'pct_s.genes', 'pct_g2m.genes', 'percent_top50_genes',
       'percent_top100_genes', 'nuc_prep_batch', 'Tank', 'Trigger',
       'Prev_tank', 'Condition', 'Status', 'Status_Long', 'Time_Day_2',
       'Behaviors_Day_2', 'Change_Length', 'Change_Mass', 'Log_11KT',
       'Average_Area_2.5x', 'Total_Slides_With_Gonad', 'Estimated_Volume_2.5x',
       'Log10_Volume', 'Percent_Testicular', 'Testicular_Estimate',
       'Log10_Testicular_Estimate', 'pct.bad_nuc', 'pct.good_nuc', 'S.Score',
       'G2M.Score', 'Phase', 'median_UMI_count', 'median_gene_count',
     

https://github.com/ventolab/CellphoneDB/blob/master/notebooks/T0_DownloadDB.ipynb

Download cellphone db

In [4]:
from IPython.display import HTML, display
from cellphonedb.utils import db_releases_utils

display(HTML(db_releases_utils.get_remote_database_versions_html()['db_releases_html_table']))

I will make a folder where these are to be downloaded
A:\CellPhoneDB 030225

In [5]:
import os
import ssl
import urllib.request

# Create directory if it doesn't exist
cpdb_version = 'v5.0.0'
cpdb_target_dir = os.path.join('A:/CellPhoneDB 030225', cpdb_version)
os.makedirs(cpdb_target_dir, exist_ok=True)

# Create a custom opener with the unverified context
ssl_context = ssl._create_unverified_context()
opener = urllib.request.build_opener(urllib.request.HTTPSHandler(context=ssl_context))
urllib.request.install_opener(opener)

# Import and download database
from cellphonedb.utils import db_utils
db_utils.download_database(cpdb_target_dir, cpdb_version)

Downloaded cellphonedb.zip into A:/CellPhoneDB 030225\v5.0.0
Downloaded complex_input.csv into A:/CellPhoneDB 030225\v5.0.0
Downloaded gene_input.csv into A:/CellPhoneDB 030225\v5.0.0
Downloaded interaction_input.csv into A:/CellPhoneDB 030225\v5.0.0
Downloaded protein_input.csv into A:/CellPhoneDB 030225\v5.0.0
Downloaded uniprot_synonyms.tsv into A:/CellPhoneDB 030225\v5.0.0\sources
Downloaded transcription_factor_input.csv into A:/CellPhoneDB 030225\v5.0.0\sources


In [6]:
cpdb_target_dir

'A:/CellPhoneDB 030225\\v5.0.0'

https://github.com/ventolab/CellphoneDB/blob/master/notebooks/T1_Method1.ipynb

I have no idea why cellphone db is making me do all this stupid file path shit, why cant I just load in an object and tell it what the objects are

meta_file_path: (mandatory) path to the meta file linking cell barcodes to cluster labels metadata.tsv.

So I'm asuming this is a tsv file with the harmony wsnn cluster labels and teh cell barcodes, I can make this now 

In [36]:
cells = {
    "cell" : obj.obs.index,
    "cell_type" : obj.obs['harmony.wnn_res0.4_clusters']
    
}

cells = pd.DataFrame(cells)
cells

filename = "A:/CellPhoneDB 030225/cluster labels metadata.tsv"

cells.to_csv(filename, sep="\t") 


I think I did it

counts_file_path: (mandatory) paths to normalized counts file (not z-transformed), either in text format or h5ad (recommended) normalised_log_counts.h5ad.

Ok so this can be the data matrix which is lognormalized 

Ok X is data matrix

In [26]:
print(obj.X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 80217767 stored elements and shape (52517, 28004)>
  Coords	Values
  (0, 12)	1.5827801206268868
  (0, 40)	1.5827801206268868
  (0, 114)	1.5827801206268868
  (0, 121)	1.5827801206268868
  (0, 187)	1.5827801206268868
  (0, 191)	2.167560461146175
  (0, 199)	1.5827801206268868
  (0, 215)	2.534126551823951
  (0, 223)	1.5827801206268868
  (0, 232)	1.5827801206268868
  (0, 243)	2.8017765706326516
  (0, 260)	1.5827801206268868
  (0, 272)	1.5827801206268868
  (0, 283)	2.167560461146175
  (0, 287)	2.167560461146175
  (0, 308)	1.5827801206268868
  (0, 353)	2.534126551823951
  (0, 364)	2.167560461146175
  (0, 366)	1.5827801206268868
  (0, 376)	1.5827801206268868
  (0, 397)	1.5827801206268868
  (0, 427)	1.5827801206268868
  (0, 452)	1.5827801206268868
  (0, 491)	1.5827801206268868
  (0, 529)	1.5827801206268868
  :	:
  (52516, 27132)	3.06864906033756
  (52516, 27142)	2.420937766945525
  (52516, 27153)	2.420937766945525
  (52516, 27160)	2.

In [19]:
from scipy.sparse import csr_matrix
#counts_file_path: (mandatory) paths to normalized counts file (not z-transformed), either in text format or h5ad (recommended) normalised_log_counts.h5ad.
obj.X = csr_matrix(obj.X)
obj.X

import hdf5plugin
obj.write_h5ad('A:/CellPhoneDB 030225/normalised_log_counts')

Assign paths to objects

In [31]:
cpdb_file_path = 'A:/CellPhoneDB 030225/v5.0.0/cellphonedb.zip'
meta_file_path = 'A:/CellPhoneDB 030225/cluster labels metadata.tsv'
counts_file_path  = 'A:/CellPhoneDB 030225/normalised_log_counts.h5ad'
out_path = 'A:/CellPhoneDB 030225/'

Inspect input files

In [27]:
metadata = pd.read_csv(meta_file_path, sep = '\t')
metadata ## do I need to remove the first column?

,Unnamed: 0,barcode_sample,cluster_idents
0,T17D_AAACAGCCAAGGCCAA-1,T17D_AAACAGCCAAGGCCAA-1,7
1,D4M_AAACAGCCACCAGCAT-1,D4M_AAACAGCCACCAGCAT-1,2
2,T19D_AAACAGCCACGTAATT-1,T19D_AAACAGCCACGTAATT-1,0
3,D4M_AAACATGCAACTAACT-1,D4M_AAACATGCAACTAACT-1,1
4,T17D_AAACATGCAAGCTTAT-1,T17D_AAACATGCAAGCTTAT-1,5
...,...,...,...
52512,T11D_TTTGTTGGTAAATTGC-1,T11D_TTTGTTGGTAAATTGC-1,15
52513,T11D_TTTGTTGGTAATTAGC-1,T11D_TTTGTTGGTAATTAGC-1,4
52514,T11D_TTTGTTGGTCCTTAGT-1,T11D_TTTGTTGGTCCTTAGT-1,0
52515,T11D_TTTGTTGGTGACATAT-1,T11D_TTTGTTGGTGACATAT-1,2


In [23]:
adata = anndata.read_h5ad(counts_file_path)
adata.shape

(52517, 28004)

In [ ]:
list(adata.obs.index).sort() == list(metadata['cell']).sort()


True

Run basic analysis

030325 - I think this will work once I change the gene names to human

In [37]:
from cellphonedb.src.core.methods import cpdb_analysis_method

cpdb_results = cpdb_analysis_method.call(
    cpdb_file_path = cpdb_file_path,           # mandatory: CellphoneDB database zip file.
    meta_file_path = meta_file_path,           # mandatory: tsv file defining barcodes to cell label.
    counts_file_path = counts_file_path,       # mandatory: normalized count matrix - a path to the counts file, or an in-memory AnnData object
    counts_data = 'hgnc_symbol',               # defines the gene annotation in counts matrix.
    score_interactions = True,                 # optional: whether to score interactions or not. 
    output_path = out_path,                    # Path to save results    microenvs_file_path = None,
    separator = '|',                           # Sets the string to employ to separate cells in the results dataframes "cellA|CellB".
    threads = 5,                               # number of threads to use in the analysis.
    threshold = 0.1,                           # defines the min % of cells expressing a gene for this to be employed in the analysis.
    result_precision = 3,                      # Sets the rounding for the mean values in significan_means.
    debug = False,                             # Saves all intermediate tables emplyed during the analysis in pkl format.
    output_suffix = None                       # Replaces the timestamp in the output files by a user defined string in the  (default: None)
)

[ ][CORE][03/03/25-11:10:51][INFO] [Non Statistical Method] Threshold:0.1 Precision:3
Reading user files...
The following user files were loaded successfully:
A:/CellPhoneDB 030225/normalised_log_counts.h5ad
A:/CellPhoneDB 030225/cluster labels metadata.tsv


AllCountsFilteredException: All counts filtered